# Problem 008

https://projecteuler.net/problem=8

In [22]:
rawDigits = "7316717653133062491922511967442657474235534919493496983520312774506326239578318016984801869478851843858615607891129494954595017379583319528532088055111254069874715852386305071569329096329522744304355766896648950445244523161731856403098711121722383113622298934233803081353362766142828064444866452387493035890729629049156044077239071381051585930796086670172427121883998797908792274921901699720888093776657273330010533678812202354218097512545405947522435258490771167055601360483958644670632441572215539753697817977846174064955149290862569321978468622482839722413756570560574902614079729686524145351004748216637048440319989000889524345065854122758866688116427171479924442928230863465674813919123162824586178664583591245665294765456828489128831426076900422421902267105562632111110937054421750694165896040807198403850962455444362981230987879927244284909188845801561660979191338754992005240636899125607176060588611646710940507754100225698315520005593572972571636269561882670428252483600823257530420752963450"


## Brute force approach

Use a for loop to iterate over each position in the given number (as a string) and for each calculate the product of the next $13$ adjacent digits. Issues are clear: 
- We have to compute $\tilde 1000$ products of 13 numbers, which can be brute forced but is incredibly inefficient. 
- We lose out previous work anytime we start a new computation. If we have the string $222222222222222$ and the next digit is a $2$, we have to recalculate the whole thing even though we know the answer.

In [23]:
from math import prod
from typing import Tuple

def cleanDigits(raw:str) -> str: #remove whitespace
    digits = "".join(raw.split())
    if not digits.isdigit():
        raise ValueError("Input must only contain digits and whitespace.")
    return digits

## Helper function
We need a way to clean up whitespace and such from a digital string. 

In [24]:
def largestProductBruteForce(rawDigits, windowSize):
    digits = cleanDigits(rawDigits)
    bestProduct = 0
    bestWindow = ""

    for i in range(len(digits) - windowSize + 1):
        window = digits[i:i+windowSize]
        currentProduct = prod(int(digit) for digit in window)

        if currentProduct > bestProduct:
            bestProduct, bestWindow = currentProduct, window
    return bestProduct, bestWindow


## Complexity

Let $n$ be the number of digits in the target number. Here, $n = 1000$. Let the number of digits we want to check be $k = 13$. 

The brute force method checks roughly $\mathcal{O}(n-k+1)$ windows. Each product would cost $k$, so the overall time complexity is $$\mathcal{O}(k(n-k+1)) \approx \mathcal{O}(nk-k^2) \approx \mathcal{O}(nk) \approx \mathcal{O}(n)$$ since we assume $n>>k$. 

The space complexity is

$$
O(1),
$$

since we only store the current highest product.



## Mathematical observation
Rather than just recalculating everything from scratch, we can use a 'sliding window' approach where we calculate the product of the first $13$ numbers, and for each subsequent product multiply it by the next adjacent number, $n_{14}$ and divide by the first integer $n_{1}$. This means our previous results at each step, and perform only 2 new calculations. 

Eagled-eyed among you may note that this is nonsensical when we see a $0$, so we instead skip ahead to the first non-zero integer past it. 


In [25]:
def largestProductSliding(rawDigits, windowSize):
    digits = cleanDigits(rawDigits)

    currentProduct = 1
    zeroCount = 0
    bestProduct = 0
    bestWindow = ""

    for right, char in enumerate(digits):
        incomingDigit = int(char)
        if incomingDigit == 0:
            zeroCount += 1
        else:
            currentProduct *= incomingDigit
        if right >= windowSize:
            outgoingDigit = int(digits[right-windowSize])
            if outgoingDigit == 0:
                zeroCount -= 1
            else:
                currentProduct//=outgoingDigit
        if right >= windowSize - 1 and zeroCount == 0:
            left = right - windowSize + 1
            window = digits[left:right+1]
            if currentProduct > bestProduct:
                bestProduct = currentProduct 
                bestWindow = window
    return bestProduct, bestWindow
    

In [26]:
bruteAnswer, bruteWindow = largestProductBruteForce(rawDigits, 13)
slidingAnswer, slidingWindow = largestProductSliding(rawDigits, 13)

assert bruteAnswer == slidingAnswer
assert bruteWindow == slidingWindow

bruteAnswer, bruteWindow

(23514624000, '5576689664895')

## Lessons learned
A relatively straightforward problem. Leetcode regulars should be able to spot how to do this after a few Easy and Medium Arrays questions. 